# Path B (tokenized): class-conditional diffusion over codec tokens

Replaces raw-waveform diffusion with diffusion over **time-domain audio tokens**. A pretrained
neural codec (DAC, 16 kHz) turns each 2-second crop into 104 frames of discrete tokens, the
diffusion model generates those tokens' embeddings, and the codec's decoder turns the tokens back
into audio. No spectrogram is involved at any point.

Data, split, crops, training loop, and sampler come from `common.py`, so this path sees exactly the
same audio crops and training recipe as Paths A, B, and C. This notebook defines only the token
transform and the model.

Set `QUICK_TEST = True` first to run the whole notebook in a few minutes.

## 1. Setup

In [ ]:
!pip install -q diffusers accelerate soundfile "transformers>=4.45"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, math, time
REPO_DIR = "/content/thesis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/1jvaldez1/audio-diffusion-synth.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import common as C
import subprocess

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Audio, display

assert hasattr(C, "CodecFrontEnd"), "common.py is an old version: pull the repo, then restart the session"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Config

Shared settings come from `C.Config`. `total_steps` stays at the default (10,000), the same budget
Paths A and B used.

In [ ]:
QUICK_TEST = False          # True: short run to check the notebook end to end

cfg = C.Config(total_steps=300, inference_steps=20) if QUICK_TEST else C.Config()
C.seed_everything(cfg.seed)

# Path B-codec settings
N_CODEBOOKS = 12            # DAC 16 kHz has 12; fewer = lower bitrate, fewer channels
CHANNELS    = (128, 256, 256)
ATTN_LEVELS = (0, 1, 2)     # levels with self-attention; (2,) mimics raw Path B's placement
DROPOUT     = 0.1
GUIDANCE    = 1.0           # keep at 1.0 for distribution metrics
N_PER_CLASS = 2 if QUICK_TEST else 16
CACHE_TOKENS = True         # encode each training clip once; False re-encodes every crop each step (slow)

TAG          = ("_cached" if CACHE_TOKENS else "") + ("_quick" if QUICK_TEST else "")
RUN_DIR      = f"{cfg.out_root}/path_b_codec{TAG}"
MEL_FRONTEND = f"{cfg.out_root}/frontend.json"                      # shared; created by the Path A notebook
CODEC_FILE   = f"{cfg.out_root}/frontend_codec_{N_CODEBOOKS}cb.json" # token statistics; created here
WAVE_FILE    = f"{cfg.out_root}/samples/path_b_codec_wave{TAG}.pt"   # generated audio
MEL_FILE     = f"{cfg.out_root}/samples/path_b_codec{TAG}.pt"        # the same audio as mel, like path_a.pt
TOKENS_FILE  = f"{cfg.out_root}/samples/path_b_codec_tokens{TAG}.pt" # the generated tokens themselves
TOKEN_CACHE  = f"{cfg.out_root}/tokens_train_{N_CODEBOOKS}cb.pt"     # every training clip's tokens, encoded once

## 3. Data and the codec front end

Same split and crops as the other paths. `C.CodecFrontEnd` wraps DAC:

* **Encode:** a crop of 32,512 samples is zero-padded to 33,280 (104 frames of 320 samples) and
  encoded into 12 tokens per frame, one per residual codebook.
* **Embed:** each token is replaced by its 8-dimensional codebook vector, giving a 96 x 104 array
  per clip. Each of the 96 channels is standardised with statistics fitted on the training split,
  the same way Paths A and C calibrate their dB window on training data only.
* **Decode:** a generated 96 x 104 array is un-standardised, each 8-dimensional chunk is snapped to
  its nearest codebook entry (so the output is real tokens), and DAC's decoder synthesises audio,
  cropped back to 32,512 samples.

**Token caching (`CACHE_TOKENS = True`).** Running DAC's encoder on every training batch costs far
more than the diffusion model itself. So every training clip is encoded once, whole, and stored
(`TOKEN_CACHE`, about 7 MB). Each training step cuts a fresh random 104-frame window out of the
stored tokens, so crop augmentation is kept. The cell after the round-trip check measures what this
changes.

The mel front end is loaded only to convert generated audio for evaluation.

In [ ]:
data = C.load_esc50(cfg)
train_split, val_split, heldout_split = C.split_by_fold(data, cfg)
train_set = C.CropDataset(train_split, cfg)
print(f"train clips: {len(train_split['waveforms'])} | val clips: {len(val_split['waveforms'])} | held-out clips: {len(heldout_split['waveforms'])}")

assert os.path.exists(MEL_FRONTEND), "run the Path A notebook first: it creates the shared mel front end"
mel_frontend = C.MelFrontEnd.load(MEL_FRONTEND, cfg, device=device)

if os.path.exists(CODEC_FILE):
    codec = C.CodecFrontEnd.load(CODEC_FILE, cfg, device=device)
else:
    codec = C.CodecFrontEnd(cfg, n_codebooks=N_CODEBOOKS, device=device)
    fit_crops, _ = C.reference_crops(train_split, cfg, k=2)        # training data only
    codec.fit(fit_crops)
    codec.save(CODEC_FILE)
assert codec.n_codebooks == N_CODEBOOKS, f"{CODEC_FILE} was fitted with {codec.n_codebooks} codebooks"

SAMPLE_SHAPE = (codec.channels, codec.frames)
print(f"clip: {cfg.clip_samples} samples ({cfg.clip_seconds:.2f} s) -> {codec.frames} frames x {N_CODEBOOKS} tokens "
      f"-> model input {SAMPLE_SHAPE[0]} x {SAMPLE_SHAPE[1]} = {SAMPLE_SHAPE[0] * SAMPLE_SHAPE[1]:,} values "
      f"(raw Path B: {cfg.clip_samples:,})")

if CACHE_TOKENS:
    if os.path.exists(TOKEN_CACHE):
        train_tokens = torch.load(TOKEN_CACHE)
    else:
        t0 = time.time()
        train_tokens = codec.encode_clips(train_split["waveforms"])
        torch.save(train_tokens, TOKEN_CACHE)
        print(f"encoded {len(train_tokens)} training clips once in {time.time() - t0:.0f} s")
    assert len(train_tokens) == len(train_split["waveforms"]), f"{TOKEN_CACHE} does not match the training split"
    train_repr_set, to_repr = C.TokenCropDataset(train_split, codec, cfg, tokens=train_tokens), codec.from_tokens
else:
    train_repr_set, to_repr = train_set, codec

batch = torch.stack([train_set[i][0] for i in range(64)])
x = to_repr(torch.stack([train_repr_set[i][0] for i in range(64)]))
print(f"model input: mean {x.mean():.3f}, std {x.std():.3f}  (standardised: about 0 and 1)")

### What the codec alone costs

Before training anything: push held-out validation crops through the codec (encode to tokens, decode
back) and compare with the originals. This is Path B-codec's representation ceiling: no generator
can do better than its own codec. The evaluation notebook measures it properly in an embedding
space; here it is a quick check in mel space, plus a listen. Bird calls carry a lot of
high-frequency content and fast transients, which is exactly what codecs tend to smear, so look at
the error per frequency band and not only the average.

In [ ]:
ref_wave, ref_labels = C.reference_crops(val_split, cfg, k=1)
rt_wave = codec.roundtrip(ref_wave)
ref_mel = torch.cat([mel_frontend(ref_wave[i : i + 64]).cpu() for i in range(0, len(ref_wave), 64)])
rt_mel  = torch.cat([mel_frontend(rt_wave[i : i + 64]).cpu() for i in range(0, len(rt_wave), 64)])

err_db = (rt_mel - ref_mel).abs() * cfg.db_range / 2                 # (N, 1, n_mels, frames)
print(f"codec round trip, mel error: mean {err_db.mean():.2f} dB, 95th percentile {err_db.flatten()[::97].quantile(0.95):.2f} dB")

plt.figure(figsize=(5, 3))
plt.plot(err_db.mean(dim=(0, 1, 3)).numpy())
plt.xlabel("mel band (low to high)"); plt.ylabel("mean abs error (dB)"); plt.title("Codec round-trip error by band")
plt.show()

for i in range(2):
    print(data["class_names"][ref_labels[i]], "- original, then codec round trip")
    display(Audio(ref_wave[i, 0].numpy(), rate=cfg.sample_rate))
    display(Audio(rt_wave[i, 0].numpy(), rate=cfg.sample_rate))

### What caching changes

Caching produces a crop's tokens from the whole clip instead of from the crop alone. Away from the
window's edges the two should be identical; near the edges the cached tokens saw neighbouring audio
the crop did not have. This cell measures both effects on validation clips:

* **Token agreement by position:** how often cached tokens equal crop-encoded tokens, frame by frame.
* **Audio difference:** both token sets decoded and compared with the original crop in mel space.
  If the cached version's error is no larger than the plain round trip's, caching costs nothing
  measurable.

In [ ]:
val_tokens = codec.encode_clips(val_split["waveforms"][:64])
val_set_tok = C.TokenCropDataset({**val_split, "waveforms": val_split["waveforms"][:64]}, codec, cfg, tokens=val_tokens)
starts = [int(v[len(v) // 2]) for v in val_set_tok.valid_idx]                # a valid frame-aligned start per clip
crops  = torch.stack([val_split["waveforms"][i, :, f * codec.hop : f * codec.hop + cfg.clip_samples] for i, f in enumerate(starts)])
cached = torch.stack([val_tokens[i, :, f : f + codec.frames].long() for i, f in enumerate(starts)])
fresh  = torch.cat([codec.tokens(crops[i : i + 32]).cpu() for i in range(0, len(crops), 32)])

agree = (cached == fresh).float()                                              # (clips, codebooks, frames)
edge = 3
print(f"token agreement, first codebook: interior {agree[:, 0, edge:-edge].mean():.1%}, "
      f"first {edge} frames {agree[:, 0, :edge].mean():.1%}, last {edge} frames {agree[:, 0, -edge:].mean():.1%}")
print(f"token agreement, all codebooks:  interior {agree[:, :, edge:-edge].mean():.1%} (later codebooks encode fine residual detail and differ more easily)")

to_mel = lambda w: torch.cat([mel_frontend(w[i : i + 64]).cpu() for i in range(0, len(w), 64)])
crop_mel = to_mel(crops)
for name, toks in (("crop encoded alone (on the fly)", fresh), ("window of whole-clip tokens (cached)", cached)):
    dec = torch.cat([codec.tokens_to_wave(toks[i : i + 32]) for i in range(0, len(toks), 32)])
    e = (to_mel(dec) - crop_mel).abs() * cfg.db_range / 2
    print(f"{name:<38} mel error vs original: mean {e.mean():.2f} dB, first/last 5 frames {e[..., :5].mean():.2f} / {e[..., -5:].mean():.2f} dB")

plt.figure(figsize=(5, 3))
plt.plot(agree.mean(dim=(0, 1)).numpy())
plt.xlabel("frame in window"); plt.ylabel("token agreement"); plt.ylim(0, 1.02)
plt.title("Cached vs crop-encoded tokens"); plt.show()

## 4. Model

A 1D UNet over the token sequence: Path B's network with its input adapted to tokens. Every
design choice and the reason for it:

| Choice | Reason |
|---|---|
| Same ResNet block as Paths A and B (GroupNorm, SiLU, scale-shift conditioning), same class embedding with a "no label" entry, same v-prediction recipe | The comparison should isolate the representation. Everything that is not the representation is held fixed. |
| The input is the 96-channel token embedding at 104 positions; no sample folding | Raw Path B folded 4 samples into channels to make 32,512 samples tractable. Tokens are already 320x shorter than the waveform, so nothing needs folding. `conv_in` maps the 96 token channels to 128 features per position. |
| Diffuse the continuous token embeddings, then snap to the nearest token | This keeps the exact training loop, noise schedule, and DDIM sampler used by every other path, so the comparison stays one-variable. A discrete-token generator (autoregressive or discrete diffusion) would change the generative model at the same time as the representation, and you could not tell which change caused a difference. Snapping at the end still yields genuine tokens, which are saved for later work. |
| Standardise each of the 96 channels with training-set statistics | The diffusion schedule assumes data with a spread comparable to unit-variance noise, the same reason raw Path B uses mu-law companding. Codebook vectors differ in scale across codebooks; without standardisation the large-scale codebooks would dominate the loss. |
| Three levels (128 / 256 / 256 channels), two halvings: 104 -> 52 -> 26 positions | 104 = 8 x 13, so the sequence halves cleanly at most three times. Two halvings already give a 26-position bottleneck where each position spans about 80 ms. With more depth the bottleneck becomes so short that halving it again would blur onsets. |
| Self-attention at every level, including full resolution | In Paths A and raw B, attention is limited to the coarsest level because thousands of positions would make it too expensive. Here the longest sequence is 104 positions, so global attention costs almost nothing at every level. Each token is 20 ms and a kernel-3 convolution sees only 60 ms, so attention is what lets every token see the whole 2-second clip. This is a real benefit of tokenising, not an unfair advantage. Set `ATTN_LEVELS = (2,)` to run the strict ablation with raw Path B's placement. |
| Head width 64 | 2 heads at 128 channels and 4 at 256. Narrower heads (raw Path B used 32) would cost more and gain little on 104 positions. |
| About 15.4M parameters | Matched to Path A (16.4M) and raw Path B (15.9M), so capacity is not the variable. The compute per step is far lower; that difference *is* the efficiency result. |
| Kernel 3, nearest-neighbour upsampling then a conv | Mirrors Paths A and B. Upsample-then-conv avoids the checkerboard artefacts of transposed convolutions. |

In [ ]:
def _norm(ch):
    return nn.GroupNorm(min(32, ch // 4), ch)


class ResBlock1D(nn.Module):
    """GroupNorm -> SiLU -> conv, twice, with the conditioning applied as scale and shift."""

    def __init__(self, in_ch, out_ch, emb_dim, dropout, kernel):
        super().__init__()
        self.norm1, self.conv1 = _norm(in_ch), nn.Conv1d(in_ch, out_ch, kernel, padding=kernel // 2)
        self.emb = nn.Linear(emb_dim, 2 * out_ch)
        self.norm2, self.conv2 = _norm(out_ch), nn.Conv1d(out_ch, out_ch, kernel, padding=kernel // 2)
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, emb):
        h = self.conv1(F.silu(self.norm1(x)))
        scale, shift = self.emb(F.silu(emb)).unsqueeze(-1).chunk(2, dim=1)
        h = self.norm2(h) * (1 + scale) + shift
        h = self.conv2(self.dropout(F.silu(h)))
        return self.skip(x) + h


class AttnBlock1D(nn.Module):
    """Self-attention over token positions, with a residual connection."""

    def __init__(self, ch, head_dim):
        super().__init__()
        self.norm = _norm(ch)
        self.attn = nn.MultiheadAttention(ch, ch // head_dim, batch_first=True)

    def forward(self, x):
        h = self.norm(x).transpose(1, 2)                      # (B, T, C)
        h, _ = self.attn(h, h, h, need_weights=False)
        return x + h.transpose(1, 2)


class TokenUNet1D(nn.Module):
    """Path B's 1D UNet adapted to a sequence of token embeddings.

    Same ResBlock / attention / conditioning design as Path B. Differences: the
    input already has `in_channels` features per position (no sample folding),
    and attention runs at every level listed in `attn_levels`.
    """

    def __init__(self, num_classes, in_channels=96, channels=(256, 384, 512), attn_levels=(1, 2),
                 layers_per_block=2, dropout=0.1, head_dim=64, kernel=3):
        super().__init__()
        self.base, emb_dim = channels[0], channels[0] * 4
        self.time_mlp = nn.Sequential(nn.Linear(self.base, emb_dim), nn.SiLU(), nn.Linear(emb_dim, emb_dim))
        self.class_emb = nn.Embedding(num_classes + 1, emb_dim)          # +1 = "no label"
        self.conv_in = nn.Conv1d(in_channels, channels[0], kernel, padding=kernel // 2)

        def res(i, o):
            return ResBlock1D(i, o, emb_dim, dropout, kernel)

        def attn(lvl, ch):
            return AttnBlock1D(ch, head_dim) if lvl in attn_levels else nn.Identity()

        last = len(channels) - 1
        self.down, skip_ch, ch = nn.ModuleList(), [channels[0]], channels[0]
        for lvl, out in enumerate(channels):
            for _ in range(layers_per_block):
                self.down.append(nn.ModuleList([res(ch, out), attn(lvl, out)]))
                ch = out; skip_ch.append(ch)
            if lvl != last:
                self.down.append(nn.Conv1d(ch, ch, 3, stride=2, padding=1)); skip_ch.append(ch)

        self.mid = nn.ModuleList([res(ch, ch), AttnBlock1D(ch, head_dim), res(ch, ch)])

        self.up = nn.ModuleList()
        for lvl, out in reversed(list(enumerate(channels))):
            for _ in range(layers_per_block + 1):
                self.up.append(nn.ModuleList([res(ch + skip_ch.pop(), out), attn(lvl, out)]))
                ch = out
            if lvl != 0:
                self.up.append(nn.Conv1d(ch, ch, 3, padding=1))            # applied after 2x nearest upsampling

        self.norm_out, self.conv_out = _norm(ch), nn.Conv1d(ch, in_channels, kernel, padding=kernel // 2)

    def forward(self, x, t, labels):
        half = self.base // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=x.device) / half)
        args = t.float()[:, None] * freqs[None]
        emb = self.time_mlp(torch.cat([args.sin(), args.cos()], dim=1).to(x.dtype)) + self.class_emb(labels)

        h = self.conv_in(x)
        skips = [h]
        for m in self.down:
            if isinstance(m, nn.ModuleList):
                h = m[1](m[0](h, emb))
            else:
                h = m(h)
            skips.append(h)
        h = self.mid[2](self.mid[1](self.mid[0](h, emb)), emb)
        for m in self.up:
            if isinstance(m, nn.ModuleList):
                h = m[1](m[0](torch.cat([h, skips.pop()], dim=1), emb))
            else:
                h = m(F.interpolate(h, scale_factor=2.0, mode="nearest"))
        return self.conv_out(F.silu(self.norm_out(h)))


def forward_fn(model, x, t, labels):
    return model(x, t, labels)

unet = TokenUNet1D(cfg.num_classes, in_channels=codec.channels, channels=CHANNELS,
                   attn_levels=ATTN_LEVELS, dropout=DROPOUT).to(device)
n_params = sum(p.numel() for p in unet.parameters())
print(f"Path B-codec params: {n_params:,}")

## 5. Train

Resumes from `RUN_DIR/last.pt` if it exists. For a run you will report, start from an empty
`RUN_DIR` and do not change `total_steps` partway through.

With `CACHE_TOKENS = True` a step costs only the diffusion model: the codec ran once, above. The
training-time line printed below is worth recording next to raw Path B's: it is the efficiency
result. (With `CACHE_TOKENS = False` each step also runs DAC's encoder, which dominates the time.)

In [ ]:
t0 = time.time()
ema, losses = C.train_diffusion(unet, forward_fn, to_repr, train_repr_set, cfg, RUN_DIR, device)
print(f"training time this session: {(time.time() - t0) / 60:.1f} min")

plt.figure(figsize=(5, 3))
plt.plot([cfg.log_every * (i + 1) for i in range(len(losses))], losses)
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Path B-codec training loss")
plt.show()

## 6. Generate and save samples

Three files are written, in the same format as the other paths:

* `WAVE_FILE`: the generated audio. The evaluation notebook reads this one.
* `MEL_FILE`: the same audio through the shared mel front end, for mel-space diagnostics.
* `TOKENS_FILE`: the generated tokens, 12 per frame.

In [ ]:
commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()

gen_labels = C.balanced_labels(cfg, N_PER_CLASS)
generated = C.sample(unet, forward_fn, SAMPLE_SHAPE, cfg, device,
                     labels=gen_labels, ema=ema, guidance=GUIDANCE)
gen_wave, gen_tokens = codec.to_wave(generated)
gen_mel = torch.cat([mel_frontend(gen_wave[i : i + 64]).cpu() for i in range(0, len(gen_wave), 64)])

# How far generated embeddings sit from the codebook entries they snap to (in standardised units).
snapped = (codec.embed_tokens(gen_tokens.to(device)) - codec.mean) / codec.std
snap_rms = (generated.to(device) - snapped).pow(2).mean().sqrt().item()

meta = dict(path="B_codec", guidance=GUIDANCE, steps=cfg.inference_steps, codec=C.CodecFrontEnd.MODEL_ID,
            n_codebooks=N_CODEBOOKS, cache_tokens=CACHE_TOKENS, channels=CHANNELS, attn_levels=ATTN_LEVELS, n_params=n_params,
            train_steps=cfg.total_steps, snap_rms=snap_rms, commit=commit)
C.save_samples(WAVE_FILE, gen_wave, gen_labels, cfg, representation="waveform", **meta)
C.save_samples(MEL_FILE, gen_mel, gen_labels, cfg, representation="mel", **meta)
C.save_samples(TOKENS_FILE, gen_tokens, gen_labels, cfg, representation="dac_tokens", **meta)

print("saved", tuple(gen_wave.shape), tuple(gen_mel.shape), tuple(gen_tokens.shape), "| code version:", commit)
print(f"snap distance (RMS, standardised units): {snap_rms:.3f}  (well below 1: the model lands near real tokens)")
print(f"waveform std {gen_wave.std():.3f} (real: {batch.std():.3f})")

## 7. Look and listen

Held-out real clips (top) against generated ones of the same class (bottom), as mel spectrograms,
then audio players. This is a sanity check only; the evaluation notebook does the scoring.

In [ ]:
show = torch.randperm(cfg.num_classes)[:5].tolist()

fig, axes = plt.subplots(2, 5, figsize=(16, 5))
for col, c in enumerate(show):
    real = ref_mel[ref_labels == c][0, 0]
    fake = gen_mel[gen_labels == c][0, 0]
    for row, (img, tag) in enumerate([(real, "real"), (fake, "generated")]):
        axes[row, col].imshow(img, origin="lower", aspect="auto", cmap="magma", vmin=-1, vmax=1)
        axes[row, col].set_title(f"{data['class_names'][c]} ({tag})")
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
plt.tight_layout(); plt.show()

for c in show[:3]:
    print(data["class_names"][c], "- real, then generated")
    display(Audio(ref_wave[ref_labels == c][0, 0].numpy(), rate=cfg.sample_rate))
    display(Audio(gen_wave[gen_labels == c][0, 0].numpy(), rate=cfg.sample_rate))

In [ ]:
# Token usage: a collapsed generator reuses a small part of each codebook.
real_tokens = torch.cat([codec.tokens(ref_wave[i : i + 32]).cpu() for i in range(0, len(ref_wave), 32)])
for name, tok in (("real (val)", real_tokens), ("generated", gen_tokens)):
    used = [len(torch.unique(tok[:, i])) for i in range(N_CODEBOOKS)]
    print(f"{name:<11} distinct tokens per codebook: {used}")

peaks = gen_wave.abs().amax(dim=(1, 2))
print(f"clips peaking at >= 0.999: {(peaks >= 0.999).float().mean():.1%} | median peak {peaks.median():.3f} (real clips peak near 0.95 or below)")